In [52]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder
from models import *

In [53]:
SEED: int = 42

## 2 (a)

Start by exploring the cardinality of the two features. How many unique values are there for each feature? What does this suggest about how you might encode them? Is one-hot encoding feasible? To prepare for the next steps, split the data into training, validation, and test sets. The validation set will allow you to use early stopping during training for the neural network models.

In [54]:
retail_df = pd.read_csv('../data/retail.csv')

retail_df.sample(n = 3)

,Quantity,StockCode,CustomerID
207777,12,23295,16053.0
76384,24,22554,12583.0
95364,12,22992,14911.0


In [55]:
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397924 entries, 0 to 397923
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Quantity    397924 non-null  int64  
 1   StockCode   397924 non-null  object 
 2   CustomerID  397924 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 9.1+ MB


In [56]:
X = retail_df[['StockCode', 'CustomerID']]
y = retail_df['Quantity'].values

log_y = np.log(y)   # Log transform y to account for heavy right tails

X_tr, X_temp, y_tr, y_temp = train_test_split(X, log_y, test_size = 0.3, random_state = SEED)
X_val, X_te, y_val, y_te   = train_test_split(X_temp, y_temp, test_size = 2/3, random_state = SEED)

In [57]:
encoder = TargetEncoder(cols = ['StockCode', 'CustomerID'])

X_tr_encoded = encoder.fit_transform(X_tr, y_tr)
X_val_encoded = encoder.transform(X_val, y_val)
X_te_encoded = encoder.transform(X_te, y_te)

X_tr_encoded = X_tr_encoded.to_numpy()
X_val_encoded = X_val_encoded.to_numpy()
X_te_encoded = X_te_encoded.to_numpy()

In [58]:
print(f"Unique Values in StockCode: {retail_df['StockCode'].nunique()}")

Unique Values in StockCode: 3665


In [59]:
print(f"Unique Values in CustomerID: {retail_df['CustomerID'].nunique()}")

Unique Values in CustomerID: 4339


In [60]:
print("Top 10 Value Counts for StockCode")
retail_df['StockCode'].value_counts().iloc[:10]

Top 10 Value Counts for StockCode


StockCode
85123A    2035
22423     1724
85099B    1618
84879     1408
47566     1397
20725     1317
22720     1159
20727     1105
POST      1099
23203     1098
Name: count, dtype: int64

In [61]:
print("Top 10 Value Counts for CustomerID")
retail_df['CustomerID'].value_counts().iloc[:10]

Top 10 Value Counts for CustomerID


CustomerID
17841.0    7847
14911.0    5677
14096.0    5111
12748.0    4596
14606.0    2700
15311.0    2379
14646.0    2080
13089.0    1818
13263.0    1677
14298.0    1637
Name: count, dtype: int64

In [62]:
(retail_df['Quantity'] > 0).all()

True

Comment: From the data, we observe that StockCode has 3665 unique categories and CustomerID has 4339 unique categories. Since StockCode and CustomerID have a large number of unique categories, one-hot encoding will be memory heavy and prone to overfitting with many spare columns, leading us to another encoding method such as target encoding.

## 2 (b)

Try target encoding. For each unique value of the feature, replace it with the mean of the target for that value (from the training set). You can use TargetEncoder from the category encoders package. Using an appropriate loss function, fit a model of your choice to the data using the target-encoded features. Evaluate the model on the test set.

In [63]:
mlp = MLP(
    d_in = X_tr_encoded.shape[1],
    d_hidden = 128,
    d_out = 1,
    bias = True,
    batch_size = 128,
    epochs = 100,
    learning_rate = 1e-3,
    device = 'cpu',
    seed = 42  
)
mlp.load_state_dict(torch.load('saved_models/mlp_b.pt'))
mlp.eval()

MLP(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [64]:
mlp_predictions = torch.from_numpy(mlp.predict(X_te_encoded)).float().view(-1, 1)
y_te_t = torch.from_numpy(y_te).float().view(-1, 1)

print(f"MLP test MSE (log): {mlp.mse_loss(mlp_predictions, y_te_t).item():.4f}")      # Note, target was transformed via log()

MLP test MSE (log): 0.5175


## 2 (c)

Try a neural network model with an embedding layer for each of the two features. Choose a reasonable embedding dimension. Fit this model to the data. Evaluate the model on the test set. How does it compare to the target encoding model?

In [65]:
ordinal_encoder = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1)
X_tr_idx = (ordinal_encoder.fit_transform(X_tr) + 1).astype(int)
X_val_idx = (ordinal_encoder.transform(X_val) + 1).astype(int)
X_te_idx = (ordinal_encoder.transform(X_te) + 1).astype(int)

n_cat = [len(c) + 1 for c in ordinal_encoder.categories_]

In [66]:
emb_mlp = EmbeddingMLP(
    d_in = X_tr_idx.shape[1],
    d_hidden = 128,
    n_cat = n_cat,
    d_embed = 16, 
    d_out = 1,
    bias = True,
    batch_size = 128,
    epochs = 100,
    learning_rate = 1e-3,
    device = 'cpu',
    seed = 42  
)

emb_mlp.load_state_dict(torch.load("saved_models/mlp_c.pt"))
emb_mlp.eval()

EmbeddingMLP(
  (embedding): ModuleList(
    (0): Embedding(3596, 16, padding_idx=0)
    (1): Embedding(4312, 16, padding_idx=0)
  )
  (network): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [67]:
emb_mlp_predictions = torch.from_numpy(emb_mlp.predict(X_te_idx)).float().view(-1, 1)
y_te_t = torch.from_numpy(y_te).float().view(-1, 1)

print(f"Embedded MLP test MSE (log): {emb_mlp.mse_loss(emb_mlp_predictions, y_te_t).item():.4f}")  

Embedded MLP test MSE (log): 1.0267


In [68]:
baseline_mse = np.mean((y_te - y_tr.mean())**2)
print(f"Baseline MSE (log): {baseline_mse:.4f}")

Baseline MSE (log): 1.4579


### Comment:

The Embedded MLP performs worse than the MLP using target-encoding features. However, the Embedded MLP still outperforms a baseline mean estimate, signalling that the model learned something.

## 2 (d)

What if we combine the two? That is, use both the target encoded features and the embeddings as input to the neural network. Fit this model to the data. Evaluate the model on the test set. How does it compare to the other two models?

### Comment:



## 2 (e)